# Assignment 01 – End-to-End ETL with a Public API

| Field | Value |
|-------|-------|
| **Student Name** | *(your name)* |
| **Assignment #** | 01 |
| **Source** | https://jsonplaceholder.typicode.com/posts |

**Goal:** extract posts from a public API, clean and enrich the data, validate the result, and save it as a CSV.

## 0 – Setup

In [ ]:
import sys, os

# Resolve project root (two levels up from this notebook)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd

from utils.extractor   import from_api
from utils.transformer import (normalize_columns, drop_duplicates_and_nulls,
                               strip_strings, add_derived_column)
from utils.loader      import to_csv, print_summary

OUT_DIR = os.path.join(PROJECT_ROOT, 'data', 'output')
os.makedirs(OUT_DIR, exist_ok=True)

print('Setup complete')

## 1 – Extract

In [ ]:
API_URL = 'https://jsonplaceholder.typicode.com/posts'
df_raw = from_api(API_URL)
df_raw.head()

## 2 – Explore

In [ ]:
print_summary(df_raw, 'Raw Posts')
df_raw.dtypes

## 3 – Clean & Transform

In [ ]:
df = df_raw.copy()

# Standardise column names
df = normalize_columns(df)

# Strip whitespace from text columns
df = strip_strings(df)

# Remove duplicate rows and fully-null columns
df = drop_duplicates_and_nulls(df)

# Derive a new column: character count of 'body'
df = add_derived_column(df, 'body_length', lambda d: d['body'].str.len())

# TODO: add your own transformation below
# e.g. keep only posts from user 1:
# from utils.transformer import filter_rows
# df = filter_rows(df, 'userid', '==', 1)

df.head()

## 4 – Validate

In [ ]:
assert not df.empty, 'DataFrame must not be empty'

required_cols = ['userid', 'id', 'title', 'body', 'body_length']
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f'Missing columns: {missing}'

assert df['id'].is_unique, 'id column must be unique'
assert df['body_length'].min() > 0, 'body_length must be positive'

print(f'Rows   : {len(df)}')
print(f'Columns: {list(df.columns)}')
print('Validation passed ✓')

## 5 – Load

In [ ]:
output_path = to_csv(df, os.path.join(OUT_DIR, 'assignment_01_posts.csv'))
print(f'Saved → {output_path}')
df[['userid', 'id', 'title', 'body_length']].head(10)

## 6 – Summary & Findings

> *Replace each bullet with your own observation.*

- The dataset contains **100 posts** from **10 users** (10 posts each).
- Post bodies range from roughly **100 to 250 characters** in length.
- No missing values were found after extraction.
- TODO: add your own findings here.